# Hard-pair specialist diagnostic

This experiment measures whether confidence-gated binary specialists improve a 5-fold GroupKFold multiclass TF-IDF baseline. It does not change the production notebook or submit anything.

In [ ]:
import glob, json, time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score

paths=glob.glob("/kaggle/input/**/train.csv", recursive=True)
print("input entries:", glob.glob("/kaggle/input/*")[:20], flush=True)
if not paths:
    paths=["/kaggle/input/name-that-llm/train.csv"]
print("using train:", paths[0], flush=True)
train=pd.read_csv(paths[0]); train["text"]=train["text"].fillna("").astype(str)
labels=sorted(train.model.unique()); label_to_i={x:i for i,x in enumerate(labels)}
y=train.model.map(label_to_i).to_numpy(); groups=train.prompt_id.to_numpy()
pairs=[("gemini-3.7-flash","gemini-3.8-flash"),("gemini-3.6-flash","gemini-3.7-flash"),("claude-fable-5","claude-opus-5"),("glm-5.3","glm-5.3-flash"),("gpt-5.6-luna","gpt-5.6-sol"),("gpt-5.6-luna","gpt-5.6-terra"),("gpt-5.6-sol","gpt-5.6-terra"),("gpt-5.5","kimi-k3")]


In [ ]:
def features(tr_text, va_text):
    A=[]; B=[]
    for analyzer,ng,maxf in [("char",(3,5),140000),("word",(1,2),90000)]:
        v=TfidfVectorizer(analyzer=analyzer,ngram_range=ng,sublinear_tf=True,min_df=2,max_features=maxf,lowercase=False,dtype=np.float32)
        A.append(v.fit_transform(tr_text)); B.append(v.transform(va_text))
    return hstack(A,format="csr"),hstack(B,format="csr")

gkf=GroupKFold(5); base_scores=[]; spec_scores=[]; records=[]; t0=time.time()
for fold,(itr,iva) in enumerate(gkf.split(train,y,groups),1):
    Xtr,Xva=features(train.text.iloc[itr].tolist(),train.text.iloc[iva].tolist())
    base=LinearSVC(C=1.5,class_weight="balanced",max_iter=4000,random_state=42).fit(Xtr,y[itr])
    dec=base.decision_function(Xva); pred=dec.argmax(1)
    order=np.argsort(dec,axis=1)[:,-2:]; margin=dec.max(1)-np.partition(dec,-2,axis=1)[:,-2]
    new=pred.copy(); applied=0
    for a,b in pairs:
        if a not in label_to_i or b not in label_to_i: continue
        ia,ib=label_to_i[a],label_to_i[b]
        mask=np.isin(order,[ia,ib]).all(1)&(margin<0.55)
        if not mask.any(): continue
        train_pair=np.isin(y[itr],[ia,ib]);
        pair=LinearSVC(C=0.8,class_weight="balanced",max_iter=3000,random_state=fold).fit(Xtr[train_pair],y[itr][train_pair])
        pp=pair.predict(Xva[mask]); new[mask]=pp; applied+=int(mask.sum())
    b=balanced_accuracy_score(y[iva],pred); s=balanced_accuracy_score(y[iva],new)
    base_scores.append(b); spec_scores.append(s); records.append(dict(fold=fold,base=b,specialist=s,delta=s-b,rows=len(iva),specialist_rows=applied))
    print(records[-1],flush=True)
res=dict(base_cv=float(np.mean(base_scores)),specialist_cv=float(np.mean(spec_scores)),delta=float(np.mean(np.array(spec_scores)-np.array(base_scores))),folds=records)
print(json.dumps(res,indent=2)); Path("/kaggle/working/hard_pair_results.json").write_text(json.dumps(res,indent=2))
